# Download Semua Data Harga Saham
Notebook ini digunakan untuk mengunduh semua data harga historis saham yang terdaftar di `List-Perusahaan/Perusahaan.csv` dan menyimpannya ke dalam folder `Dataset/` secara otomatis.

In [1]:
import yfinance as yf
import pandas as pd
import os
import time

def historical_price(Code_Saham):
    path_csv = f"./Dataset/{Code_Saham}.csv"
    
    # Buat folder Dataset otomatis di root jika belum ada
    os.makedirs(os.path.dirname(path_csv), exist_ok=True)
    
    # Download data 3 tahun terakhir
    saham = yf.download(Code_Saham, period="3y", interval="1d", progress=False)
    if saham.empty:
        raise ValueError("Data kosong atau ticker tidak ditemukan di yfinance")
        
    # Fix multi-index header
    saham.columns = [col[0] if isinstance(col, tuple) else col for col in saham.columns]

    # Jadikan Date kolom biasa
    saham.reset_index(inplace=True)

    # Convert ke datetime
    saham["Date"] = pd.to_datetime(saham["Date"])

    # Gabungkan dengan file lama jika sudah ada
    if os.path.exists(path_csv):
        old = pd.read_csv(path_csv)
        old["Date"] = pd.to_datetime(old["Date"])
        df = pd.concat([old, saham])
        df = df.drop_duplicates(subset="Date", keep="last")
        df = df.sort_values(by="Date").reset_index(drop=True)
    else:
        df = saham

    # Simpan kembali ke CSV
    df.to_csv(path_csv, index=False)
    return len(df)


In [2]:
# 1. Load list perusahaan
list_perusahaan_path = "./List-Perusahaan/Perusahaan.csv"
if not os.path.exists(list_perusahaan_path):
    raise FileNotFoundError(f"File {list_perusahaan_path} tidak ditemukan! Pastikan Anda sudah menjalankan scraping list perusahaan terlebih dahulu.")

df_perusahaan = pd.read_csv(list_perusahaan_path)
tickers = df_perusahaan['Ticker_YF'].dropna().unique().tolist()
total_tickers = len(tickers)

print(f"Total saham yang akan didownload: {total_tickers}")


Total saham yang akan didownload: 825


In [3]:
# 2. Loop download dengan error handling
success_count = 0
failed_tickers = []

print("Memulai proses download...\n")

for i, ticker in enumerate(tickers, 1):
    try:
        # Beri jeda 0.5 detik agar tidak terkena rate limit dari API Yahoo Finance
        time.sleep(0.5)
        
        total_rows = historical_price(ticker)
        success_count += 1
        print(f"[{i}/{total_tickers}] {ticker}: BERHASIL ({total_rows} baris data)")
        
    except Exception as e:
        failed_tickers.append((ticker, str(e)))
        print(f"[{i}/{total_tickers}] {ticker}: GAGAL - {str(e)}")

print("\n=======================================")
print(f"Proses Selesai!")
print(f"Berhasil: {success_count}/{total_tickers}")
print(f"Gagal: {len(failed_tickers)}/{total_tickers}")
print("=======================================")

if failed_tickers:
    print("\nDaftar ticker yang gagal:")
    for t, err in failed_tickers:
        print(f"- {t}: {err}")


Memulai proses download...

[1/825] AMRT.JK: BERHASIL (717 baris data)
[2/825] MLPL.JK: BERHASIL (717 baris data)
[3/825] TAPG.JK: BERHASIL (717 baris data)
[4/825] MPPA.JK: BERHASIL (717 baris data)
[5/825] MIDI.JK: BERHASIL (717 baris data)
[6/825] LAPD.JK: BERHASIL (717 baris data)
[7/825] RANC.JK: BERHASIL (717 baris data)
[8/825] DMND.JK: BERHASIL (717 baris data)
[9/825] HERO.JK: BERHASIL (717 baris data)
[10/825] PCAR.JK: BERHASIL (717 baris data)
[11/825] EPMT.JK: BERHASIL (717 baris data)
[12/825] AMMS.JK: BERHASIL (717 baris data)
[13/825] SDPC.JK: BERHASIL (717 baris data)
[14/825] KMDS.JK: BERHASIL (717 baris data)
[15/825] BUAH.JK: BERHASIL (717 baris data)
[16/825] DAYA.JK: BERHASIL (717 baris data)
[17/825] WICO.JK: BERHASIL (720 baris data)
[18/825] MAMIP.JK: BERHASIL (715 baris data)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: GWAA.JK"}}}
$GWAA.JK: possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['GWAA.JK']: possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")


[19/825] GWAA.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[20/825] INDF.JK: BERHASIL (717 baris data)
[21/825] ICBP.JK: BERHASIL (717 baris data)
[22/825] AALI.JK: BERHASIL (717 baris data)
[23/825] JPFA.JK: BERHASIL (717 baris data)
[24/825] MYOR.JK: BERHASIL (717 baris data)
[25/825] CPIN.JK: BERHASIL (717 baris data)
[26/825] RLCO.JK: BERHASIL (134 baris data)
[27/825] LSIP.JK: BERHASIL (717 baris data)
[28/825] GZCO.JK: BERHASIL (717 baris data)
[29/825] FORE.JK: BERHASIL (293 baris data)
[30/825] BWPT.JK: BERHASIL (717 baris data)
[31/825] JARR.JK: BERHASIL (717 baris data)
[32/825] SIMP.JK: BERHASIL (717 baris data)
[33/825] ULTJ.JK: BERHASIL (717 baris data)
[34/825] ASHA.JK: BERHASIL (717 baris data)
[35/825] WMUU.JK: BERHASIL (717 baris data)
[36/825] CPRO.JK: BERHASIL (717 baris data)
[37/825] IKAN.JK: BERHASIL (717 baris data)
[38/825] CLEO.JK: BERHASIL (717 baris data)
[39/825] DSNG.JK: BERHASIL (717 baris data)
[40/825] COCO.JK: BERHASIL (717 baris data

$JELI.JK: possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['JELI.JK']: possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")


[119/825] JELI.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[120/825] UNVR.JK: BERHASIL (717 baris data)
[121/825] BRRC.JK: BERHASIL (350 baris data)
[122/825] NANO.JK: BERHASIL (717 baris data)
[123/825] MBTO.JK: BERHASIL (717 baris data)
[124/825] MRAT.JK: BERHASIL (717 baris data)
[125/825] KINO.JK: BERHASIL (717 baris data)
[126/825] VICI.JK: BERHASIL (717 baris data)
[127/825] UCID.JK: BERHASIL (717 baris data)
[128/825] MSJA.JK: BERHASIL (586 baris data)
[129/825] EURO.JK: BERHASIL (717 baris data)
[130/825] TCID.JK: BERHASIL (717 baris data)
[131/825] FLMC.JK: BERHASIL (717 baris data)
[132/825] KPAS.JK: BERHASIL (488 baris data)
[133/825] GGRM.JK: BERHASIL (717 baris data)
[134/825] HMSP.JK: BERHASIL (717 baris data)
[135/825] WIIM.JK: BERHASIL (717 baris data)
[136/825] ITIC.JK: BERHASIL (717 baris data)


...

[500/825] BORN.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[501/825] ALII.JK: BERHASIL (566 baris data)
[502/825] SEMA.JK: BERHASIL (717 baris data)
[503/825] JSKY.JK: BERHASIL (715 baris data)
[504/825] IOTF.JK: BERHASIL (651 baris data)
[505/825] MTDL.JK: BERHASIL (717 baris data)
[506/825] NINE.JK: BERHASIL (717 baris data)
[507/825] LUCK.JK: BERHASIL (717 baris data)
[508/825] PTSN.JK: BERHASIL (717 baris data)
[509/825] ZYRX.JK: BERHASIL (717 baris data)
[510/825] AXIO.JK: BERHASIL (717 baris data)
[511/825] MENN.JK: BERHASIL (720 baris data)
[512/825] GLVA.JK: BERHASIL (717 baris data)
[513/825] CHIP.JK: BERHASIL (717 baris data)
[514/825] KLBF.JK: BERHASIL (717 baris data)
[515/825] SIDO.JK: BERHASIL (717 baris data)
[516/825] KAEF.JK: BERHASIL (717 baris data)
[517/825] PYFA.JK: BERHASIL (717 baris data)
[518/825] TSPC.JK: BERHASIL (717 baris data)
[519/825] INAF.JK: BERHASIL (720 baris data)
[520/825] SOHO.JK: BERHASIL (717 baris data)
[521/825] MDLA.JK: BE

$NPII.JK: possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['NPII.JK']: possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")


[740/825] NPII.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[741/825] ASII.JK: BERHASIL (717 baris data)
[742/825] BNBR.JK: BERHASIL (719 baris data)
[743/825] BHIT.JK: BERHASIL (717 baris data)
[744/825] FOLK.JK: BERHASIL (693 baris data)
[745/825] ZBRA.JK: BERHASIL (720 baris data)
[746/825] KING.JK: BERHASIL (718 baris data)
[747/825] UNTR.JK: BERHASIL (717 baris data)
[748/825] PIPA.JK: BERHASIL (717 baris data)
[749/825] IMPC.JK: BERHASIL (717 baris data)
[750/825] HEXA.JK: BERHASIL (717 baris data)
[751/825] LABA.JK: BERHASIL (717 baris data)
[752/825] NTBK.JK: BERHASIL (717 baris data)
[753/825] SMIL.JK: BERHASIL (717 baris data)
[754/825] KUAS.JK: BERHASIL (717 baris data)
[755/825] SINI.JK: BERHASIL (717 baris data)
[756/825] TOTO.JK: BERHASIL (717 baris data)
[757/825] PTMP.JK: BERHASIL (717 baris data)
[758/825] HOPE.JK: BERHASIL (717 baris data)
[759/825] KOBX.JK: BERHASIL (717 baris data)
[760/825] MARK.JK: BERHASIL (717 baris data)
[761/825] ARNA.JK: BE

$ASIA.JK: possibly delisted; no price data found  (period=3y)

1 Failed download:
['ASIA.JK']: possibly delisted; no price data found  (period=3y)


[785/825] ASIA.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[786/825] GIAA.JK: BERHASIL (717 baris data)
[787/825] BIRD.JK: BERHASIL (717 baris data)
[788/825] ASSA.JK: BERHASIL (717 baris data)
[789/825] IMJS.JK: BERHASIL (717 baris data)
[790/825] WEHA.JK: BERHASIL (717 baris data)
[791/825] CMPP.JK: BERHASIL (717 baris data)
[792/825] TAXI.JK: BERHASIL (717 baris data)
[793/825] TRJA.JK: BERHASIL (717 baris data)
[794/825] LRNA.JK: BERHASIL (717 baris data)
[795/825] BPTR.JK: BERHASIL (717 baris data)
[796/825] SAFE.JK: BERHASIL (717 baris data)
[797/825] HELI.JK: BERHASIL (717 baris data)
[798/825] WBSA.JK: BERHASIL (59 baris data)
[799/825] PJHB.JK: BERHASIL (156 baris data)
[800/825] SMDR.JK: BERHASIL (717 baris data)
[801/825] LAJU.JK: BERHASIL (717 baris data)
[802/825] BLOG.JK: BERHASIL (239 baris data)
[803/825] TMAS.JK: BERHASIL (717 baris data)
[804/825] JAYA.JK: BERHASIL (717 baris data)
[805/825] SDMU.JK: BERHASIL (717 baris data)
[806/825] KLAS.JK: BER